### Create a copy of the initial catalogue with an additional ‘type’ column that can take 4 values: 
    - FRI
    - FRII
    - Not Classifiable (UC)
    - Unsure (US)

In [1]:
import os
from tkinter import *
from PIL import Image, ImageTk
from astropy.io import fits
from astropy.table import Table, Column
import numpy as np 

In [2]:
sources = [file for file in os.listdir('Final_facet_subimages') if file.endswith('.png')] #List of the sources to classify
sources_path = [os.path.join('Final_facet_subimages', i) for i in sources] #List of the path of the sources to classify

In [6]:
'''A little application to help with classification
It displays the pngs one after the other, for each you can choose "FRI, FRII, Not classifiable, Not sure" and it saves it in a list'''

type = []

root = Tk()
root.title('Source Classification')

max_size = (1530, 510)
sources_img = [ImageTk.PhotoImage(Image.open(i).resize(max_size, Image.LANCZOS)) for i in sources_path]

i = 0
label = Label(image=sources_img[i])
label.grid(row=1, column=0, columnspan=4)

status = Label(root, text=f'{i+1}/{len(sources)}', anchor=E)
status.grid(row=3, column=0, columnspan=4, sticky=W+E)

ID = Label(root, text=f'{sources[i]}')
ID.grid(row=0, column=0, columnspan=4, sticky=W+E)

def update_img():
    label.configure(image=sources_img[i])
    label.image = sources_img[i]
    status.config(text=f'{i+1}/{len(sources)}')
    ID.config(text=f'{sources[i]}')
    
    
def FRI(event=None):
    global i
    global type
    
    type.append((sources[i][:-4], 'FRI'))
    i += 1
    if i == len(sources):
        root.destroy()

    else:
        update_img()

def FRII(event=None):
    global i
    global type
    
    type.append((sources[i][:-4], 'FRII'))
    i += 1
    if i == len(sources):
        root.destroy()

    else:
        update_img()

def NC(event=None):
    global i
    global type
    
    type.append((sources[i][:-4], 'NC'))
    i += 1
    if i == len(sources):
        root.destroy()

    else:
        update_img()

def US(event=None):
    global i
    global type
    
    type.append((sources[i][:-4], 'US'))
    i += 1
    if i == len(sources):
        root.destroy()

    else:
        update_img()

FRI_b = Button(root, text="FRI", command=FRI)
FRII_b = Button(root, text="FRII", command=FRII)
NC_b = Button(root, text="NC", command=NC)
US_b = Button(root, text="US", command=US)

FRI_b.grid(row=2, column=0, pady=5)
FRII_b.grid(row=2, column=1, pady=5)
NC_b.grid(row=2, column=2, pady=5)
US_b.grid(row=2, column=3, pady=5)

root.mainloop()

In [9]:
with fits.open('candidate_sample_new.fits') as hdul:
    '''Saves the list in the catalogue in fits format'''
    
    cat = Table(hdul[1].data)
    header = hdul[1].header

    ID = [source[0]for source in type]
    types = [source[1]for source in type]

    cat_types = []

    for id in cat['Source_ID']:
        index = ID.index(id)
        cat_types.append(types[index])

    cat.add_column(Column(cat_types, name='type'))

    new_hdul = fits.HDUList([hdul[0], fits.BinTableHDU(cat, header=header)])
    new_hdul.writeto('cat_classified_v0.fits', overwrite=True)